# Sieci neuronowe i deep learning - Lab 5

## CNN i transfer learning

PyTorch.

Pracujemy na **CIFAR-10**. Celem jest porownanie CNN trenowanego od zera z modelem wykorzystujacym backbone pretrained na ImageNet.

Do punktacji bazowym zbiorem jest CIFAR-10. Wlasny/trudniejszy zbior moze byc rozszerzeniem, ale musi zachowac zgodny preprocessing, format etykiet i liczbe klas w ostatniej warstwie.

## Zadanie 1

- Zaladuj CIFAR-10 i pokaz przykladowe obrazy.
- Opcjonalna podmiana datasetu nie zastepuje punktowanego wariantu bazowego bez zgody prowadzacego.
- Zbuduj CNN trenowany od zera.
- Wytrenuj model i narysuj krzywe uczenia.
- Porownaj accuracy treningowe i walidacyjne.

In [ ]:
# Konfiguracja techniczna.
# Ta komorka jest gotowa, bo dotyczy glownie cache danych i importow.

import os
from pathlib import Path


def configure_torch_cache():
    candidates = [
        Path("../dane/torchvision"),
        Path("laboratoria/dane/torchvision"),
        Path("dane/torchvision"),
        Path("/content/dane/torchvision"),
    ]
    for candidate in candidates:
        if candidate.parent.exists():
            candidate.mkdir(parents=True, exist_ok=True)
            model_cache = candidate / "models"
            model_cache.mkdir(parents=True, exist_ok=True)
            os.environ["TORCH_HOME"] = str(model_cache.resolve())
            return candidate.resolve(), model_cache.resolve()
    fallback = Path("dane/torchvision")
    fallback.mkdir(parents=True, exist_ok=True)
    model_cache = fallback / "models"
    model_cache.mkdir(parents=True, exist_ok=True)
    os.environ["TORCH_HOME"] = str(model_cache.resolve())
    return fallback.resolve(), model_cache.resolve()


DATA_ROOT, TORCH_HOME = configure_torch_cache()

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.models import MobileNet_V2_Weights

from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split


RANDOM_STATE = 42
BATCH_SIZE = 128
IMAGE_SIZE = 96
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_STATE)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


In [ ]:
# Zadanie 1a: dane i transformacje.

# TODO:
# - pobierz CIFAR-10,
# - wydziel train/validation ze zbioru treningowego,
# - przygotuj osobne transformacje dla baseline, modelu regularyzowanego i transfer learning.

# Zbiór treningowy CIFAR-10: 50 000 obrazów 32x32 RGB, 10 klas po 5000 obrazów.
# Wczytujemy go tu bez transformacji - potrzebne są tylko etykiety do podziału i surowe piksele.
base_train = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True)

all_indices = np.arange(len(base_train))
# 10% (5000 obrazów) idzie na walidację; stratify -> w walidacji jest dokładnie po 500 obrazów z każdej klasy.
# Zbiór testowy (osobne 10 000 obrazów) ruszamy dopiero w wyniku końcowym.
train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.1,
    stratify=base_train.targets,
    random_state=RANDOM_STATE,
)

# Średnia i odchylenie każdego kanału RGB liczone tylko na części treningowej (bez podglądania walidacji).
# base_train.data ma kształt (50000, 32, 32, 3) i typ uint8.
train_pixels = base_train.data[train_indices].astype("float32") / 255.0
CIFAR_MEAN = train_pixels.mean(axis=(0, 1, 2)).tolist()
CIFAR_STD = train_pixels.std(axis=(0, 1, 2)).tolist()
del train_pixels

# Baseline: tylko zamiana na tensor i normalizacja, bez żadnej augmentacji.
scratch_train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
scratch_eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Model regularyzowany: losowe przesunięcie (wycinek 32x32 z obrazu dopełnionego 4 pikselami z każdej strony)
# i losowe odbicie w poziomie. Obie zmiany nie zmieniają klasy obiektu (samolot odbity dalej jest samolotem).
# Walidacja używa scratch_eval_transform - bez augmentacji.
regularized_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Transfer learning: MobileNetV2 uczył się na ImageNet (obrazy ok. 224x224), więc powiększamy CIFAR do 96x96
# i normalizujemy statystykami ImageNet - takimi samymi, jakie widział backbone podczas treningu.
transfer_train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
transfer_eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

NUM_WORKERS = 4  # procesy wczytujące dane równolegle z treningiem na GPU


def make_loaders(train_transform, val_transform, batch_size=BATCH_SIZE):
    # TODO: utworz DataLoadery dla train i validation, uzywajac Subset.
    # Dwie instancje CIFAR10 różnią się tylko transformacją; Subset wybiera z nich nasze indeksy.
    train_dataset = Subset(datasets.CIFAR10(root=DATA_ROOT, train=True, transform=train_transform), train_indices)
    val_dataset = Subset(datasets.CIFAR10(root=DATA_ROOT, train=True, transform=val_transform), val_indices)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader


print("urządzenie:", DEVICE)
print("train:", len(train_indices), "| validation:", len(val_indices))
print("obrazów na klasę w train:", np.bincount(np.array(base_train.targets)[train_indices]))
print("średnia RGB (train):", np.round(CIFAR_MEAN, 3), "| odchylenie RGB:", np.round(CIFAR_STD, 3))

In [ ]:
# Zadanie 1b: podglad danych.

# TODO: pokaz siatke przykladowych obrazow z podpisami klas.
# Warto sprawdzic, czy etykiety zgadzaja sie z obrazami.

# base_train nie ma transformacji, więc Subset zwraca pary (obraz PIL, etykieta) - od razu do wyświetlenia.
preview_dataset = Subset(base_train, train_indices)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, idx in zip(axes.ravel(), range(16)):
    image, label = preview_dataset[idx]
    ax.imshow(image)
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")
plt.suptitle("CIFAR-10: przykładowe obrazy treningowe (32x32)")
plt.tight_layout()
plt.show()

In [ ]:
# Funkcje pomocnicze.
# Tu nie powinno byc logiki modelu, tylko wspolna petla treningowa i narzedzia do porownan.


def plot_history(history, title):
    # TODO: narysuj loss i accuracy dla train/validation.
    raise NotImplementedError


def print_score_table(rows):
    # TODO: wypisz porownanie modeli w czytelnej tabeli.
    raise NotImplementedError


def count_parameters(model):
    # TODO: policz tylko trenowalne parametry.
    raise NotImplementedError


def run_epoch(model, loader, loss_fn, optimizer=None):
    # TODO:
    # - obsluz tryb train/eval,
    # - przenies dane na DEVICE,
    # - policz loss i accuracy,
    # - wykonaj backward/step tylko w treningu.
    raise NotImplementedError


def train_torch_model(model, name, train_loader, val_loader, epochs=10, learning_rate=0.001, weight_decay=0.0):
    # TODO: pelna petla treningowa z historia metryk i wierszem wyniku.
    raise NotImplementedError

In [ ]:
# Zadanie 1c: CNN od zera.


class ScratchCNN(nn.Module):
    def __init__(self, dropout_rate=0.0):
        super().__init__()
        # TODO:
        # - zaprojektuj kilka blokow Conv2d,
        # - dodaj normalizacje/aktywacje/pooling,
        # - dodaj klasyfikator dla 10 klas,
        # - dropout_rate ma miec realny efekt.
        raise NotImplementedError

    def forward_features(self, x):
        # TODO: zwroc aktywacje z kilku warstw, zeby dac sie potem zwizualizowac.
        raise NotImplementedError

    def forward(self, x):
        # TODO: forward klasyfikatora.
        raise NotImplementedError


torch_rows = []
scratch_train_loader, scratch_val_loader = make_loaders(scratch_train_transform, scratch_eval_transform)

# TODO: wytrenuj baseline CNN od zera i dodaj jego wynik do torch_rows.
scratch_model = ...
scratch_model, scratch_history, scratch_row = ...
...

plot_history(...)
print_score_table(torch_rows)

## Zadanie 2

- Wyswietl filtry pierwszej warstwy konwolucyjnej modelu bazowego.
- Wyswietl mapy aktywacji wybranej warstwy konwolucyjnej dla jednego obrazu testowego.
- Krotko opisz, co widac w aktywacjach.

In [ ]:
# Zadanie 2: filtry i aktywacje.

# TODO:
# - wybierz pierwsza warstwe Conv2d,
# - wyswietl kilka filtrow,
# - pobierz jedna probke walidacyjna,
# - uzyj forward_features i wyswietl aktywacje wybranej warstwy.

conv_layer = ...
conv1_weights = ...

...

sample_batch = ...
activations = ...

...

## Zadanie 3

- Zastosuj metody poprawy generalizacji, np. augmentacje, dropout, L2/weight decay, batch normalization.
- Wytrenuj nowy model.
- Porownaj wynik walidacyjny z modelem bazowym.

In [ ]:
# Zadanie 3: generalizacja.

# TODO:
# - zbuduj drugi model CNN,
# - zastosuj co najmniej dwie techniki poprawy generalizacji,
# - uzasadnij wybrane wartosci dropout/weight_decay/augmentacji,
# - porownaj wynik z baseline.

regularized_train_loader, regularized_val_loader = ...
regularized_model = ...
regularized_model, regularized_history, regularized_row = ...

torch_rows.append(regularized_row)
plot_history(...)
print_score_table(torch_rows)

## Zadanie 4

- Uzyj modelu pretrained na ImageNet, np. MobileNetV2.
- Zamroz backbone i wytrenuj nowa glowice klasyfikacyjna dla CIFAR-10.
- Odblokuj ostatnie warstwy backbone'u i wykonaj krotki fine-tuning z malym learning rate.
- Porownaj transfer learning z modelem uczonym od zera.

In [ ]:
# Zadanie 4: transfer learning.

# TODO:
# - utworz MobileNetV2 pretrained na ImageNet,
# - przygotuj preprocessing dla CIFAR-10,
# - zamroz features i wymien classifier,
# - wytrenuj nowa glowice,
# - odblokuj wybrane koncowe bloki i wykonaj fine-tuning z malym learning rate,
# - dopisz oba wyniki do torch_rows.

transfer_train_loader, transfer_val_loader = ...

weights = ...
transfer_model = ...

transfer_model, transfer_history, transfer_row = ...
torch_rows.append(transfer_row)

transfer_model, fine_tune_history, fine_tune_row = ...
torch_rows.append(fine_tune_row)

plot_history(...)
plot_history(...)
print_score_table(torch_rows)

## Wynik koncowy

- Uzyj go jako koncowego raportu: wybierz najlepszy model na podstawie walidacji.
- Dopiero potem policz wynik na zbiorze testowym i wyswietl macierz pomylek.
- Wyniku testowego nie uzywaj do dobierania hiperparametrow.


In [ ]:
# Wynik koncowy.

# TODO:
# - upewnij sie, ze wiersze w torch_rows przechowuja model_object oraz test_transform,
# - wybierz najlepszy model na podstawie validation accuracy,
# - policz accuracy i classification_report na tescie,
# - pokaz macierz pomylek,
# - napisz 2-3 zdania komentarza pod komorka.

test_loader = ...
best_row = ...
best_model = ...

test_pred = ...
test_true = ...

print(...)
print(...)

...
